# Co-expression conservation between human and mouse PT (notebook 97)

**The question.** Species comparisons so far compare levels (confounded by the probe panels) or shapes (a quarter of
each gradient shared, no pathway-level species program). A level-free third readout: does a gene keep the same
co-expression partners in both species (Crow et al. 2022's co-expression conservation)? Is it reproducible, confirmed
in independent atlases, non-redundant with the gradient comparison, and does it find pathway-level rewiring?

**The frozen protocol** is `docs/results/pt-coexpression-conservation.md` (137fd92; amended before any computation in
590b6c5). Per gene: within-species reproducibility W_m and W_h, cross-species conservation X (mean of four specimen
pairs) and rewiring D = (W_m + W_h) / 2 − X, on a total and a position-residual network; the same in the Census atlases.

**Sections.** 1 setup and inputs; 2 universe and networks; 3 scores; 4 Q1–Q5; 5 verdicts; 6 figures; 7 reading.

## 1 · Setup and inputs

In [ ]:
import argparse
import json
import os
import random
import sys
import time
from pathlib import Path

NOTEBOOK_START = time.perf_counter()
starts = [Path.cwd()] + ([Path(__file__).resolve().parent] if '__file__' in globals() else [])
project = next(p for s in starts for p in (s, *s.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
EXPORT = results_root / 'package_pseudospace' / 'hvm' / 'pt_units.h5ad'
ORTHOLOGS = results_root / 'minimal_pt_scfates' / 'ortholog_map_used.csv'
T71 = results_root / 'pt_pathways_package_pseudospace' / 'tables'
T90 = results_root / 'pt_continuous_axis_experiments' / 'tables'
ATLAS = data_root / 'external' / 'census_pt_cells'
output = results_root / 'pt_coexpression_conservation'
tables, figures = output / 'tables', output / 'figures'
for folder in (tables, figures):
    folder.mkdir(parents=True, exist_ok=True)

NOTEBOOK_LOGIC_VERSION = '97.coexpression.1'
SEED = 97
FINGERPRINT = 'efafc9ac2e99'
MOUSE, HUMAN = ['Ctrl1A2', 'Ctrl1A4'], ['HUK1_COR1', 'HUK1_MED1']
SPECIMENS = MOUSE + HUMAN
OUR_DETECTION, ATLAS_DETECTION, MIN_NUCLEI = .10, .05, 300
K_PARTNERS, N_BINS, N_PERM = 10, 20, 1000
Q1_MIN, Q2_MIN_RHO, Q2_P, Q3_MAX_RHO = .7, .2, .01, .5
SET_SIZES = (10, 300)
REQUIRED = [EXPORT, ORTHOLOGS, T71 / 'universe_human_vs_mouse.csv', T71 / 'gene_sets_human_vs_mouse.json',
            T90 / 'd_end_contrasts.csv.gz', ATLAS / 'human_pt_cells.h5ad', ATLAS / 'mouse_pt_cells.h5ad']
for path in REQUIRED:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path} (run notebooks 70, 71 and 90; fetch the Census PT cells)')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata as ad
from scipy import sparse
from scipy.stats import mannwhitneyu, spearmanr
from IPython.display import display

from pseudospace.coexpression_conservation import aggregate, binned_residuals, conservation, spearman_network
from pseudospace.package_inputs import hvm_inputs, read_pt_units
from pseudospace.pathway_pipelines import camera_pr, over_representation

MM = 1 / 25.4
INK, MUTED, LIGHT = '#0b0b0b', '#52514e', '#d9d8d4'
CAT_COLOR = {'conserved': '#1b8a5a', 'reversed': '#d55e00', 'mouse only': '#2a78d6', 'human only': '#9C93D3', 'none': LIGHT}
plt.rcParams.update({
    'font.family': ['Liberation Sans', 'DejaVu Sans'], 'font.size': 6.5, 'axes.titlesize': 6.5,
    'axes.titleweight': 'bold', 'axes.labelsize': 6.5, 'xtick.labelsize': 6, 'ytick.labelsize': 6,
    'legend.fontsize': 6, 'legend.frameon': False, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': .6, 'xtick.major.width': .6, 'ytick.major.width': .6, 'xtick.major.size': 2.5,
    'ytick.major.size': 2.5, 'lines.linewidth': 1.0, 'pdf.fonttype': 42, 'svg.fonttype': 'none', 'savefig.dpi': 300})
pd.set_option('display.width', 240, 'display.max_columns', 40, 'display.max_colwidth', 80, 'display.max_rows', 80)


def save(fig, stem):
    for suffix in ('png', 'pdf'):
        fig.savefig(figures / f'{stem}.{suffix}', bbox_inches='tight')

### 1.1 · Inputs and guards (not cached)

- G1: the human vs mouse export has notebook 70's pass-1 fingerprint, and `hvm_inputs` (mt out) rebuilds notebook
  71's universe.

In [ ]:
guard_log = []
orth = pd.read_csv(ORTHOLOGS)
MT = sorted(g for g in orth.mouse_symbol.astype(str) if g.startswith('mt-'))
units = read_pt_units(EXPORT, 'hvm')
assert units.uns['provenance']['fingerprints']['pass1']['membership_sha1'] == FINGERPRINT, 'G1: fingerprint'
D = hvm_inputs(units, data_root, orth, exclude_genes=MT)
del units
U71 = pd.read_csv(T71 / 'universe_human_vs_mouse.csv').gene.astype(str).tolist()
assert D['genes'] == U71, "G1: the universe differs from notebook 71's"
guard_log.append(f"G1: export fingerprint {FINGERPRINT}; hvm_inputs rebuilds 71's universe ({len(U71):,} genes)")
print('\n'.join('· ' + g for g in guard_log))
SPEC = np.asarray(D['specimen']).astype(str)
POS = np.asarray(D['position'], float)
Y = D['Y']
Y = Y.toarray() if sparse.issparse(Y) else np.asarray(Y, float)
COUNTS = sparse.csr_matrix(D['counts'])
GENES = pd.Index(D['genes'])
print({s: int((SPEC == s).sum()) for s in SPECIMENS})

atlas = {}
for species in ('mouse', 'human'):
    a = ad.read_h5ad(ATLAS / f'{species}_pt_cells.h5ad')
    counts = sparse.csr_matrix(a.X)
    donors = a.obs.donor_id.astype(str)
    keep_donors = donors.value_counts()[lambda v: v >= MIN_NUCLEI].index.tolist()
    keep = donors.isin(keep_donors).to_numpy()
    atlas[species] = {'counts': counts[keep], 'donor': donors.to_numpy()[keep], 'genes': pd.Index(a.var_names.astype(str)),
                      'raw_sum': a.obs.raw_sum.to_numpy(float)[keep], 'donors': sorted(keep_donors, key=lambda d: -int((donors == d).sum()))}
    del a
    print(species, f'{len(keep_donors)} donors with ≥ {MIN_NUCLEI} nuclei, {int(keep.sum()):,} nuclei')

## 2 · Universe G and networks

In [ ]:
det = {s: np.asarray((COUNTS[SPEC == s] > 0).mean(axis=0)).ravel() for s in SPECIMENS}
ours_ok = np.all([det[s] >= OUR_DETECTION for s in SPECIMENS], axis=0)
atlas_ok = np.ones(len(GENES), bool)
for species, a in atlas.items():
    idx = a['genes'].get_indexer(GENES)
    frac = np.zeros(len(GENES))
    frac[idx >= 0] = np.asarray((a['counts'][:, idx[idx >= 0]] > 0).mean(axis=0)).ravel()
    atlas_ok &= (idx >= 0) & (frac >= ATLAS_DETECTION)
G = GENES[ours_ok & atlas_ok].tolist()
gpos = GENES.get_indexer(G)
print(f'G: {len(G):,} genes (ours ≥ 10% in all four specimens: {int(ours_ok.sum()):,}; atlases ≥ 5%: {int(atlas_ok.sum()):,})')

t0 = time.perf_counter()
NET = {'total': {}, 'residual': {}}
for s in SPECIMENS:
    m = SPEC == s
    X = Y[m][:, gpos]
    NET['total'][s] = spearman_network(X)
    NET['residual'][s] = spearman_network(binned_residuals(X, POS[m], n_bins=N_BINS))
print(f'Our networks {time.perf_counter() - t0:.0f} s')


def atlas_networks(species):
    a = atlas[species]
    idx = a['genes'].get_indexer(G)
    out = {}
    for d in a['donors']:
        m = a['donor'] == d
        block = a['counts'][m][:, idx].toarray()
        out[d] = spearman_network(np.log1p(block / a['raw_sum'][m, None] * 1e4))
    return out


t0 = time.perf_counter()
ANET = {sp: atlas_networks(sp) for sp in atlas}
AGG = {sp: aggregate(list(nets.values())) for sp, nets in ANET.items()}
HALVES = {sp: (aggregate([ANET[sp][d] for d in atlas[sp]['donors'][0::2]]),
               aggregate([ANET[sp][d] for d in atlas[sp]['donors'][1::2]])) for sp in atlas}
print(f'Atlas networks {time.perf_counter() - t0:.0f} s')
SUMMARY = pd.DataFrame([{'network': f'ours · {layer} · {s}', 'genes': len(G), 'units or nuclei': int((SPEC == s).sum()),
                         'median |r| off-diagonal': float(np.median(np.abs(NET[layer][s][np.triu_indices(len(G), 1)])))}
                        for layer in NET for s in SPECIMENS]
                       + [{'network': f'atlas · {sp} · {len(atlas[sp]["donors"])} donors', 'genes': len(G),
                           'units or nuclei': int(len(atlas[sp]['donor']))} for sp in atlas])
SUMMARY.to_csv(tables / 'networks_summary.csv', index=False)
display(SUMMARY.round(3))

## 3 · Per-gene scores

In [ ]:
SCORES = pd.DataFrame(index=pd.Index(G, name='gene'))
for layer, nets in NET.items():
    wm = conservation(nets[MOUSE[0]], nets[MOUSE[1]], k=K_PARTNERS)
    wh = conservation(nets[HUMAN[0]], nets[HUMAN[1]], k=K_PARTNERS)
    x = np.mean([conservation(nets[m], nets[h], k=K_PARTNERS) for m in MOUSE for h in HUMAN], axis=0)
    SCORES[f'W_m {layer}'], SCORES[f'W_h {layer}'], SCORES[f'X {layer}'] = wm, wh, x
    SCORES[f'D {layer}'] = (wm + wh) / 2 - x
SCORES['X atlas'] = conservation(AGG['mouse'], AGG['human'], k=K_PARTNERS)
SCORES['W atlas mouse'] = conservation(*HALVES['mouse'], k=K_PARTNERS)
SCORES['W atlas human'] = conservation(*HALVES['human'], k=K_PARTNERS)
SCORES['W atlas'] = SCORES[['W atlas mouse', 'W atlas human']].mean(axis=1)
SCORES['D atlas'] = SCORES['W atlas'] - SCORES['X atlas']
END = pd.read_csv(T90 / 'd_end_contrasts.csv.gz').set_index('mouse')
END.index = END.index.astype(str)
SCORES['E mouse'] = END['E mouse'].reindex(G)
SCORES['E human'] = END['E human'].reindex(G)
SCORES['gradient divergence'] = (SCORES['E mouse'] - SCORES['E human']).abs()
SCORES['category'] = END['category'].reindex(G).fillna('none')
SCORES.to_csv(tables / 'genes.csv')
display(SCORES.describe().T.round(3))

## 4 · Q1–Q5

In [ ]:
Q1 = pd.DataFrame({layer: {'median W_m': SCORES[f'W_m {layer}'].median(), 'median W_h': SCORES[f'W_h {layer}'].median(),
                           'median X': SCORES[f'X {layer}'].median(), 'median D': SCORES[f'D {layer}'].median(),
                           'share D > 0': float((SCORES[f'D {layer}'] > 0).mean())} for layer in NET}).T
Q1.loc['atlas'] = {'median W_m': SCORES['W atlas mouse'].median(), 'median W_h': SCORES['W atlas human'].median(),
                   'median X': SCORES['X atlas'].median(), 'median D': SCORES['D atlas'].median(),
                   'share D > 0': float((SCORES['D atlas'] > 0).mean())}
Q1.to_csv(tables / 'q1_reproducibility.csv')
display(Q1.round(3))

rng = np.random.default_rng(SEED)
Q2 = []
for layer in NET:
    for a_col, b_col in ((f'D {layer}', 'D atlas'), (f'X {layer}', 'X atlas')):
        a, b = SCORES[a_col].to_numpy(), SCORES[b_col].to_numpy()
        rho = spearmanr(a, b)[0]
        null = np.array([spearmanr(a, rng.permutation(b))[0] for _ in range(N_PERM)])
        Q2.append({'layer': layer, 'pair': f'{a_col} vs {b_col}', 'ρ': rho, 'null 99%': float(np.quantile(null, .99)),
                   'p': float((1 + (null >= rho).sum()) / (1 + N_PERM))})
Q2 = pd.DataFrame(Q2)
Q2.to_csv(tables / 'q2_external.csv', index=False)
display(Q2.round(4))

Q3 = []
for layer in NET:
    ok = SCORES['gradient divergence'].notna()
    rho = spearmanr(SCORES.loc[ok, f'D {layer}'], SCORES.loc[ok, 'gradient divergence'])[0]
    by = SCORES[ok].groupby('category')[f'D {layer}'].agg(['size', 'median'])
    rev, con = SCORES.loc[SCORES.category.eq('reversed'), f'D {layer}'], SCORES.loc[SCORES.category.eq('conserved'), f'D {layer}']
    p = mannwhitneyu(rev, con, alternative='two-sided').pvalue if len(rev) and len(con) else np.nan
    Q3.append({'layer': layer, 'genes with a gradient': int(ok.sum()), 'ρ(D, gradient divergence)': rho,
               'p reversed vs conserved': p, **{f'median D {c} (n)': f'{r["median"]:.3f} ({int(r["size"])})' for c, r in by.iterrows()}})
Q3 = pd.DataFrame(Q3)
Q3.to_csv(tables / 'q3_gradients.csv', index=False)
display(Q3.T)

SETS = json.loads((T71 / 'gene_sets_human_vs_mouse.json').read_text())
SETS_G = {k: sorted(set(v) & set(G)) for k, v in SETS.items()}
SETS_G = {k: v for k, v in SETS_G.items() if SET_SIZES[0] <= len(v) <= SET_SIZES[1]}
Q4 = []
for layer in NET:
    ours = camera_pr(SCORES[f'D {layer}'], SETS_G).set_index('pathway_id')
    ext = camera_pr(SCORES['D atlas'], SETS_G).set_index('pathway_id')
    t = pd.DataFrame({'layer': layer, 'n': ours.n, 'direction': ours.direction, 'fdr': ours.fdr,
                      'atlas direction': ext.direction.reindex(ours.index), 'atlas fdr': ext.fdr.reindex(ours.index)})
    t['rewired (both)'] = t.direction.eq('Up') & t['atlas direction'].eq('Up') & (t.fdr <= .05) & (t['atlas fdr'] <= .05)
    t['conserved (both)'] = t.direction.eq('Down') & t['atlas direction'].eq('Down') & (t.fdr <= .05) & (t['atlas fdr'] <= .05)
    Q4.append(t.rename_axis('set').reset_index())
Q4 = pd.concat(Q4)
Q4.to_csv(tables / 'q4_sets.csv', index=False)
Q4SUM = []
for layer, t in Q4.groupby('layer'):
    n = len(t)
    for kind, d in (('rewired', 'Up'), ('conserved', 'Down')):
        a = int(((t.direction == d) & (t.fdr <= .05)).sum())
        b = int(((t['atlas direction'] == d) & (t['atlas fdr'] <= .05)).sum())
        Q4SUM.append({'layer': layer, 'kind': kind, 'sets': n, 'ours': a, 'atlas': b, 'both': int(t[f'{kind} (both)'].sum()),
                      'expected by chance': a * b / n if n else np.nan})
Q4SUM = pd.DataFrame(Q4SUM)
display(Q4SUM)
for layer in NET:
    t = Q4[Q4.layer.eq(layer)]
    print(layer, 'rewired in both:', '; '.join(t.set[t['rewired (both)']].str.split('::').str[1].head(25)))
    print(layer, 'conserved in both:', '; '.join(t.set[t['conserved (both)']].str.split('::').str[1].head(25)))

### Q5 · Tubule-level programs beyond position (residual layer, descriptive)

In [ ]:
import igraph as ig

meanres = np.mean([NET['residual'][s] for s in MOUSE], axis=0)
A = meanres.copy()
np.fill_diagonal(A, -np.inf)
top = np.argpartition(-A, K_PARTNERS, axis=1)[:, :K_PARTNERS]
edges = {(min(i, j), max(i, j)) for i in range(len(G)) for j in top[i]}
graph = ig.Graph(n=len(G), edges=sorted(edges))
random.seed(SEED)
part = graph.community_leiden(objective_function='modularity', resolution=1., n_iterations=-1)
MODULE = pd.Series(part.membership, index=G)
Q5 = []
for mod, members in MODULE.groupby(MODULE):
    genes = members.index.tolist()
    if len(genes) < 10:
        continue
    ora = over_representation(genes, SETS_G, G).sort_values('p').head(3)
    Q5.append({'module': int(mod), 'genes': len(genes), 'mean X residual': float(SCORES.loc[genes, 'X residual'].mean()),
               'mean W_m residual': float(SCORES.loc[genes, 'W_m residual'].mean()),
               'top genes': ', '.join(genes[:12]),
               'top sets (fdr)': '; '.join(f"{p.split('::')[1]} ({f:.2g})" for p, f in zip(ora.pathway_id, ora.fdr))})
Q5 = pd.DataFrame(Q5).sort_values('genes', ascending=False)
Q5.to_csv(tables / 'q5_modules.csv', index=False)
display(Q5.head(20))

## 5 · Verdicts

In [ ]:
q1 = Q1.loc['total', 'median W_m'] >= Q1_MIN
q2row = Q2[Q2.layer.eq('total') & Q2.pair.eq('D total vs D atlas')].iloc[0]
q2 = q2row['ρ'] >= Q2_MIN_RHO and q2row['p'] <= Q2_P
q3 = abs(Q3[Q3.layer.eq('total')]['ρ(D, gradient divergence)'].iloc[0]) <= Q3_MAX_RHO
q4 = bool((Q4SUM[Q4SUM.layer.eq('total')].both > Q4SUM[Q4SUM.layer.eq('total')]['expected by chance']).any())
VERD = pd.DataFrame([
    {'question': 'Q1 reproducible neighbourhoods (median W_m ≥ 0.7, total)', 'holds': bool(q1), 'value': f"{Q1.loc['total', 'median W_m']:.3f}"},
    {'question': 'Q2 rewiring confirmed in the atlases (ρ ≥ 0.2, p ≤ 0.01)', 'holds': bool(q2), 'value': f"ρ {q2row['ρ']:.3f}, p {q2row['p']:.3g}"},
    {'question': 'Q3 non-redundant with gradients (|ρ| ≤ 0.5)', 'holds': bool(q3), 'value': f"{Q3[Q3.layer.eq('total')]['ρ(D, gradient divergence)'].iloc[0]:.3f}"},
    {'question': 'Q4 more replicated sets than chance', 'holds': q4, 'value': Q4SUM[Q4SUM.layer.eq('total')][['kind', 'both', 'expected by chance']].round(2).to_dict('records')},
])
VERD['method verdict'] = 'useful species method' if (q1 and q2 and (q3 or q4)) else 'not shown useful'
VERD.to_csv(tables / 'verdicts.csv', index=False)
display(VERD)

### 5.1 · Post hoc: genes with reproducible neighbourhoods in both species

Added after the run showed W bimodal (Q1 fails at the median, 0.670, while the upper quartile is 0.96). The measure is
only meaningful for genes whose neighbourhood reproduces within each species: W_m ≥ 0.7 and W_h ≥ 0.7 (total). Q2 and
Q3 on that subset, and its most and least rewired genes. Not judged.

In [ ]:
REP = SCORES[(SCORES['W_m total'] >= Q1_MIN) & (SCORES['W_h total'] >= Q1_MIN)]
a, b = REP['D total'].to_numpy(), REP['D atlas'].to_numpy()
rho_rep = spearmanr(a, b)[0]
null_rep = np.array([spearmanr(a, rng.permutation(b))[0] for _ in range(N_PERM)])
POST = pd.DataFrame([{'genes': len(REP), 'ρ(D, D atlas)': rho_rep, 'null 99%': float(np.quantile(null_rep, .99)),
                      'ρ(D, gradient divergence)': spearmanr(REP['D total'], REP['gradient divergence'])[0],
                      'median X total': REP['X total'].median(), 'median X atlas': REP['X atlas'].median()}])
POST.to_csv(tables / 'post_hoc_reproducible.csv', index=False)
display(POST.round(3))
cols = ['W_m total', 'W_h total', 'X total', 'D total', 'D atlas', 'E mouse', 'E human', 'category']
both = REP[(REP['D total'] > REP['D total'].quantile(.9)) & (REP['D atlas'] > REP['D atlas'].quantile(.9))]
print('Rewired in both (top decile of D in ours and in the atlases):', len(both))
display(both.sort_values('D total', ascending=False)[cols].head(30).round(3))
kept = REP[(REP['D total'] < REP['D total'].quantile(.1)) & (REP['D atlas'] < REP['D atlas'].quantile(.1))]
print('Conserved in both (bottom decile):', len(kept))
display(kept.sort_values('D total')[cols].head(30).round(3))
pd.concat([both.assign(group='rewired in both'), kept.assign(group='conserved in both')]).to_csv(tables / 'post_hoc_gene_lists.csv')

## 6 · Figures

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(130 * MM, 45 * MM), sharey=True)
for ax, layer in zip(axes, NET):
    for col, color in ((f'W_m {layer}', '#2a78d6'), (f'W_h {layer}', '#9C93D3'), (f'X {layer}', INK)):
        ax.hist(SCORES[col], bins=np.linspace(0, 1, 51), histtype='step', color=color, label=col.split()[0])
    ax.axvline(.5, color=MUTED, lw=.5, ls=':')
    ax.set_title(layer)
    ax.set_xlabel('partner AUROC')
axes[0].set_ylabel('genes')
axes[0].legend(loc='upper left')
save(fig, '97a_conservation_distributions')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(120 * MM, 55 * MM))
for ax, layer in zip(axes, NET):
    ax.scatter(SCORES['D atlas'], SCORES[f'D {layer}'], s=2, color=INK, alpha=.4, lw=0, rasterized=True)
    ax.axhline(0, color=MUTED, lw=.5); ax.axvline(0, color=MUTED, lw=.5)
    ax.set_xlabel('rewiring D, atlases')
    ax.set_ylabel(f'rewiring D, ours ({layer})')
fig.tight_layout()
save(fig, '97b_rewiring_external')
plt.show()

fig, ax = plt.subplots(figsize=(70 * MM, 50 * MM))
cats = ['conserved', 'reversed', 'mouse only', 'human only', 'none']
data = [SCORES.loc[SCORES.category.eq(c), 'D total'].dropna() for c in cats]
parts = ax.violinplot([d for d in data if len(d)], showmedians=True)
ax.set_xticks(range(1, sum(len(d) > 0 for d in data) + 1), [c for c, d in zip(cats, data) if len(d)], rotation=30, ha='right')
ax.axhline(0, color=MUTED, lw=.5)
ax.set_ylabel('rewiring D (total)')
save(fig, '97c_rewiring_by_gradient_category')
plt.show()

fig, ax = plt.subplots(figsize=(60 * MM, 55 * MM))
ours_cam = camera_pr(SCORES['D total'], SETS_G).set_index('pathway_id')
ext_cam = camera_pr(SCORES['D atlas'], SETS_G).set_index('pathway_id')
from scipy.stats import norm
zo = np.where(ours_cam.direction.eq('Up'), 1, -1) * norm.isf(ours_cam.p / 2)
ze = (np.where(ext_cam.direction.eq('Up'), 1, -1) * norm.isf(ext_cam.p / 2))
ze = pd.Series(ze, index=ext_cam.index).reindex(ours_cam.index)
ax.scatter(ze, zo, s=3, color=INK, alpha=.5, lw=0)
ax.axhline(0, color=MUTED, lw=.5); ax.axvline(0, color=MUTED, lw=.5)
ax.set_xlabel('set rewiring z, atlases'); ax.set_ylabel('set rewiring z, ours')
save(fig, '97d_set_rewiring')
plt.show()

## 7 · How the protocol was read, and the run record

In [ ]:
READING = pd.DataFrame([
    ('amended before any computation (590b6c5)', 'Gradient divergence per gene is |E mouse − E human| (the frozen "1 − cosine" is undefined for one contrast per gene).'),
    ('reading', "Our expression is hvm_inputs' Y: log1p(count / library × 10⁴) with the library over the measured genes (mt included in the library, excluded from the universe)."),
    ('reading', "Atlas expression: log1p(count / raw_sum × 10⁴) per nucleus, raw_sum being the export's per-nucleus total."),
    ('reading', 'Atlas halves alternate donors ordered by nucleus count; the residual layer has no atlas twin (atlas nuclei carry no position), so Q2 judges the total layer.'),
    ('post hoc', '5.1: Q2 and Q3 on genes reproducible in both species (W ≥ 0.7), and the genes in the top or bottom decile of D in both ours and the atlases. Not judged.'),
    ('reading', 'Q5 modules: Leiden (modularity, resolution 1, seed 97) on the top-10-partner graph of the mean mouse residual network; modules < 10 genes are not listed; ORA against G.'),
], columns=['kind', 'what'])
READING.to_csv(tables / 'how_the_protocol_was_read.csv', index=False)
RUNREC = {'notebook': '97_pt_coexpression_conservation', 'logic_version': NOTEBOOK_LOGIC_VERSION, 'seed': SEED,
          'guards': guard_log, 'genes G': len(G), 'atlas donors': {sp: len(atlas[sp]['donors']) for sp in atlas},
          'verdicts': VERD[['question', 'holds']].to_dict('records'), 'method verdict': VERD['method verdict'].iloc[0],
          'seconds': round(time.perf_counter() - NOTEBOOK_START, 1)}
(output / 'run_record.json').write_text(json.dumps(RUNREC, indent=2, default=str))
print(json.dumps(RUNREC, indent=2, default=str))